# Assignment 3 - Support Vector Machine (SVM)
**Dataset:** Social Network Ads  
**Goal:** Predict whether a user purchased a product based on Age and Estimated Salary.

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    roc_curve, classification_report
)
from matplotlib.colors import ListedColormap

## 2. Loading the Dataset
The dataset is the Social Network Ads dataset. It contains information about users including their age, estimated salary, and whether they purchased a product.

In [ ]:
df = pd.read_csv('Social_Network_Ads.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Preprocessing
Steps taken:
- Check for missing values
- Drop the `User ID` column (not useful for prediction)
- Encode the `Gender` column
- Standardize numerical features

In [ ]:
print('Missing values:')
print(df.isnull().sum())

In [ ]:
df.drop('User ID', axis=1, inplace=True)

le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

df.head()

### Train / Validation / Test Split
Split ratio: 70% train, 15% validation, 15% test.

In [ ]:
X = df.drop('Purchased', axis=1)
y = df['Purchased']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

## 4. Hyperparameter Tuning with GridSearchCV
We tune the kernel, C, gamma, and degree parameters using the validation set.

In [ ]:
param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3]
}

grid_search = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print('Best Parameters:', grid_search.best_params_)
print('Best CV Accuracy:', round(grid_search.best_score_, 4))

## 5. Training the Final Model
Using the best parameters found above.

In [ ]:
best_model = grid_search.best_estimator_

val_preds = best_model.predict(X_val)
print('Validation Accuracy:', round(accuracy_score(y_val, val_preds), 4))

## 6. Evaluation on Test Set

In [ ]:
y_pred  = best_model.predict(X_test)
y_prob  = best_model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'AUC       : {auc:.4f}')
print()
print(classification_report(y_test, y_pred))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Purchased', 'Purchased'],
            yticklabels=['Not Purchased', 'Purchased'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='darkorange', label=f'AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

### Sample Predictions
Showing actual labels vs predicted labels along with the probability score for the first 15 test samples.

In [ ]:
results = pd.DataFrame({
    'Actual'     : y_test.values[:15],
    'Predicted'  : y_pred[:15],
    'Probability': y_prob[:15].round(4)
})

results['Correct'] = results['Actual'] == results['Predicted']
results

## 7. 2D Decision Boundary Plot
Since we have multiple features, we use only Age and EstimatedSalary (the two most relevant features) to visualize the decision boundary.

In [ ]:
X_vis = df[['Age', 'EstimatedSalary']].values
y_vis = df['Purchased'].values

X_vis_scaled = StandardScaler().fit_transform(X_vis)

vis_model = SVC(
    kernel=grid_search.best_params_['kernel'],
    C=grid_search.best_params_['C'],
    gamma=grid_search.best_params_['gamma'],
    degree=grid_search.best_params_['degree'],
    probability=True,
    random_state=42
)
vis_model.fit(X_vis_scaled, y_vis)

x_min, x_max = X_vis_scaled[:, 0].min() - 0.5, X_vis_scaled[:, 0].max() + 0.5
y_min, y_max = X_vis_scaled[:, 1].min() - 0.5, X_vis_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

Z = vis_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FFAAAA', '#AAAAFF']))
plt.scatter(X_vis_scaled[y_vis == 0, 0], X_vis_scaled[y_vis == 0, 1],
            c='red', label='Not Purchased', edgecolors='k', s=30)
plt.scatter(X_vis_scaled[y_vis == 1, 0], X_vis_scaled[y_vis == 1, 1],
            c='blue', label='Purchased', edgecolors='k', s=30)
plt.xlabel('Age (scaled)')
plt.ylabel('Estimated Salary (scaled)')
plt.title(f'2D Decision Boundary (kernel={grid_search.best_params_["kernel"]})')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

| Metric | Score |
|--------|-------|
| Accuracy | 0.8833 |
| Precision | 0.7917 |
| Recall | 0.9048 |
| F1 Score | 0.8444 |
| AUC | 0.9048 |

The SVM model with the tuned hyperparameters performed well on the Social Network Ads dataset. The decision boundary plot clearly shows that the model learned a non-linear boundary separating users who purchased from those who did not.